# Cat vs Dog Classification

A hands-on walkthrough of binary image classification with PyTorch.

- **Task:** binary image classification — is this a dog or a cat?
- **Data:** 2,000 training / 400 test images, 100x100 RGB, balanced classes.
  The notebook prefers your local `data/cats_vs_dogs/` CSVs, can download the
  exact dataset from Kaggle, and has a small public fallback for Colab.
- **Model:** a small CNN — two convolutions, two max-pools, one 64-unit dense
  layer, one output.

> Runs locally with `uv run jupyter lab` and in Colab via the badge in the
> folder README. The setup cell installs the extras Colab lacks.

## What this notebook covers

1. Getting the dataset from local CSVs, Kaggle, or a public fallback.
2. Loading 100x100 images into PyTorch tensors.
3. Reshaping NHWC → NCHW and scaling pixels (never labels).
4. Building the CNN with `nn.Module`.
5. A manual training loop: `zero_grad → forward → loss → backward → step`.
6. Loss/accuracy curves, confusion matrix, and predictions.
7. Saving and reloading the weights.
8. A high-level recap of what actually matters.

> The dataset is **not** committed to git. `data/` is ignored, and the exact
> Kaggle copy is downloaded on demand.

## PyTorch image conventions

Five rules cover almost every bug in an image pipeline:

| Topic | Rule |
|---|---|
| Image layout | PyTorch uses **channels first**: `(N, C, H, W)` |
| Pixel range | Floats in `[0, 1]`, not `0-255` |
| Labels | Integer class indices (`torch.long`) or float `0/1` for binary BCE |
| Loss input | Raw **logits**, not probabilities |
| Flatten size | `nn.Linear` needs the exact `C*H*W` size written out |

A batch adds the leading `N`. One RGB image is `(3, 100, 100)`; a batch of 32 is
`(32, 3, 100, 100)`.

For binary classification the model returns **one logit**. `nn.BCEWithLogitsLoss`
applies the sigmoid internally, so the model itself has no sigmoid layer.

## Labels are not pixels

A very common mistake is to scale the whole array:

```python
X_train = X_train / 255.   # good: pixels become [0, 1]
y_train = y_train / 255.   # bad: labels become 0 and 0.0039
```

Only **pixels** should be divided by 255. Labels must stay exactly `0` and `1`;
otherwise the network almost never sees a positive target, and the sigmoid
collapses toward one class. If a binary model predicts the same class for every
image, check the label range first.

The label convention used throughout:

| Label | Class |
|---|---|
| `0` | dog |
| `1` | cat |

The public fallback is loaded with `ImageFolder`, which sorts folder names
alphabetically (`cats = 0`, `dogs = 1`). We flip it with a `target_transform`
so both data paths use the same `0 = dog, 1 = cat` convention.

The prediction grid uses `random.sample` (not `randint(0, len(...))`) so it can
never index past the end of the dataset.

## Environment setup

The cell below:

- installs `torchinfo` and `kagglehub` in Colab if they are missing (no-op locally);
- imports the libraries;
- seeds Python, NumPy and PyTorch;
- picks CUDA, Apple MPS or CPU;
- sets the shared `data/` folder, `IMG_SIZE`, `BATCH_SIZE` and `EPOCHS`.

In [ ]:
# Setup: install missing Colab packages, import libraries, seed, and pick a device.
import sys

if "google.colab" in sys.modules:
    import subprocess

    for package in ("torchinfo", "kagglehub"):
        try:
            __import__(package)
        except ModuleNotFoundError:
            subprocess.run(
                [sys.executable, "-m", "pip", "-q", "install", package],
                check=True,
            )

import json
import os
import random
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from torchinfo import summary

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


def repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


DATA_DIR = repo_root() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR = DATA_DIR / "cats_vs_dogs"

# 100x100 is a good speed/detail trade-off for CPU training.
IMG_SIZE = 100
BATCH_SIZE = 32
EPOCHS = 10
NUM_WORKERS = 2 if device.type == "cuda" else 0

# 0 = dog, 1 = cat.
CLASS_NAMES = ["dog", "cat"]

print(f"PyTorch {torch.__version__} | device: {device}")
print(f"Data folder : {DATA_DIR}")
print(f"Local CSVs  : {CSV_DIR}")
print(f"Image size  : {IMG_SIZE}x{IMG_SIZE} | batch {BATCH_SIZE} | epochs {EPOCHS}")

**Takeaways**

- `torchinfo` and `kagglehub` are the only extras; everything else ships with
  the base environment (and Colab).
- `(C, H, W)` is the PyTorch image layout; a batch adds a leading `N`.
- Set the seed and choose the device once, at the top.
- `EPOCHS` lives here so every experiment is easy to change. Set `EPOCHS = 1`
  for a quick smoke test.

## Get the data

### The dataset

The local copy is four CSVs:

| File | Size | Contents |
|---|---|---|
| `input.csv` | ~1.4 GB | 2,000 images × 30,000 floats (100×100×3, flattened) |
| `labels.csv` | ~4 KB | 2,000 labels: `0 = dog`, `1 = cat` |
| `input_test.csv` | ~287 MB | 400 images × 30,000 floats |
| `labels_test.csv` | ~1.2 KB | 400 labels |

That is **~1.7 GB** of text for what is really 2,400 small images. The first
cell below reshapes the floats back to `(N, 100, 100, 3)` and caches a compact
`uint8` `.npz` so reruns are fast.

### Why this is not in git

- GitHub rejects any single file larger than **100 MB**. `input.csv` is 1.4 GB,
  so a normal push is refused.
- Git LFS does not solve it for free: the free tier is 1 GB of storage and 2 GB
  of bandwidth per month, and this dataset is already 1.7 GB.
- Datasets do not belong in a notebook repository. `data/` and `*.npz` are in
  `.gitignore`, so the CSVs and cache cannot be committed by accident.
- Never commit `kaggle.json` or API keys either.

### Where the notebook gets the data (in priority order)

| Priority | Source | Size | Credentials | Exact? |
|---|---|---|---|---|
| 1 | `data/cats_vs_dogs/*.csv` (your local copy) | ~1.7 GB | none | ✅ exact |
| 2 | [Kaggle: `tarekmohamedadel/cats-and-dogs-images`](https://www.kaggle.com/datasets/tarekmohamedadel/cats-and-dogs-images) | ~1.8 GB | optional for public data; recommended | ✅ exact |
| 3 | Public `cats_and_dogs_filtered` image mirror | ~66 MB | none | ❌ same task, different images |

The Kaggle dataset is Apache 2.0. The public fallback has 2,000 training and
1,000 validation images, so its test score is not directly comparable to the
400-image local test set.

### Getting the data in Google Colab

**Option A — exact data from Kaggle (recommended)**

1. On Kaggle, open **Settings → API → Create New Token**. This downloads
   `kaggle.json`.
2. In Colab, either:
   - click the **key icon** in the left sidebar and add secrets named
     `KAGGLE_USERNAME` and `KAGGLE_KEY`, or
   - upload `kaggle.json` and move it to `~/.kaggle/kaggle.json`.
3. Run the data cell. It calls `kagglehub.dataset_download(...)`, finds the four
   CSVs, and caches the reshaped arrays. The download is cached after the first
   run.

`kagglehub` can often download public datasets anonymously, but credentials are
recommended: they avoid rate limits and work if Kaggle changes anonymous access.

**Option B — no credentials**

Set `USE_KAGGLE = False` in the next cell. The notebook downloads the 66 MB
public image fallback instead, so the whole notebook still runs end-to-end.

**Option C — use your exact local copy in Colab**

Upload the four CSVs to `data/cats_vs_dogs/`, or mount Google Drive and point
`CSV_DIR` at it. The notebook always prefers `data/cats_vs_dogs/` when all four
files are present.

In [ ]:
# Pick the data source: local CSVs first, then Kaggle, then the public fallback.
KAGGLE_DATASET = "tarekmohamedadel/cats-and-dogs-images"
KAGGLE_PAGE = f"https://www.kaggle.com/datasets/{KAGGLE_DATASET}"
PUBLIC_ZIP_URL = (
    "https://github.com/MLBasic/cats_and_dogs_filtered/raw/master/"
    "cats_and_dogs_filtered.zip"
)

# Set to False to skip the ~1.8 GB Kaggle download and use the 66 MB fallback.
USE_KAGGLE = True

CSV_FILES = {
    "train_images": CSV_DIR / "input.csv",
    "train_labels": CSV_DIR / "labels.csv",
    "test_images": CSV_DIR / "input_test.csv",
    "test_labels": CSV_DIR / "labels_test.csv",
}
HAS_LOCAL_CSV = all(path.exists() for path in CSV_FILES.values())


def get_kaggle_credentials():
    """Read Kaggle credentials from env, ~/.kaggle/kaggle.json, or Colab secrets."""
    username = os.environ.get("KAGGLE_USERNAME")
    key = os.environ.get("KAGGLE_KEY")

    token_path = Path.home() / ".kaggle" / "kaggle.json"
    if (not username or not key) and token_path.exists():
        token = json.loads(token_path.read_text())
        username = username or token.get("username")
        key = key or token.get("key")

    if "google.colab" in sys.modules:
        try:
            from google.colab import userdata

            username = username or userdata.get("KAGGLE_USERNAME")
            key = key or userdata.get("KAGGLE_KEY")
        except Exception:
            # Secret missing or Colab userdata unavailable: anonymous may still work.
            pass

    return (username, key) if username and key else None


def find_csv(root, filename):
    matches = sorted(Path(root).rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} not found under {root}")
    return matches[0]


def download_kaggle_csvs():
    import kagglehub

    credentials = get_kaggle_credentials()
    if credentials is not None:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = credentials
        print("Kaggle credentials found.")
    else:
        print("No Kaggle credentials found; trying anonymous access (public dataset).")

    target = DATA_DIR / "kaggle_cats_dogs"
    target.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {KAGGLE_DATASET} (~1.8 GB, cached after the first run) ...")
    path = kagglehub.dataset_download(KAGGLE_DATASET, output_dir=str(target))
    return Path(path)


def ensure_csvs(root):
    """KaggleHub normally extracts automatically; handle a leftover zip too."""
    root = Path(root)
    if root.is_file() and root.suffix == ".zip":
        with zipfile.ZipFile(root) as zf:
            zf.extractall(root.parent)
        root = root.parent

    names = ("input.csv", "labels.csv", "input_test.csv", "labels_test.csv")
    if any(not list(root.rglob(name)) for name in names):
        for archive in root.rglob("*.zip"):
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(archive.parent)

    for name in names:
        find_csv(root, name)
    return root


def download_and_extract(url, folder_name, root=DATA_DIR):
    target = root / folder_name
    if target.exists():
        print(f"Already available: {target}")
        return target

    zip_path = root / Path(url).name
    if not zip_path.exists():
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, zip_path)
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(root)
    return target


CSV_ROOT = None
if HAS_LOCAL_CSV:
    CSV_ROOT = CSV_DIR
    DATA_SOURCE = "local CSVs"
    print(f"Using local CSVs from {CSV_DIR}")
elif USE_KAGGLE:
    try:
        CSV_ROOT = ensure_csvs(download_kaggle_csvs())
        DATA_SOURCE = "Kaggle CSVs"
    except Exception as exc:
        print(f"Kaggle download unavailable: {exc}")
        print("Falling back to the 66 MB public image dataset.")
        CSV_ROOT = None
        DATA_SOURCE = "public image fallback"
else:
    DATA_SOURCE = "public image fallback"

if CSV_ROOT is None:
    print("Using the public image fallback (no credentials needed).")

print(f"\nData source: {DATA_SOURCE}")

**Takeaways**

- The notebook has **one training pipeline** and three ways to feed it.
- Local CSVs are fastest and exact; Kaggle gives the same data on a fresh machine;
  the public fallback keeps Colab runnable without credentials.
- A 1.4 GB file is well past GitHub's 100 MB per-file limit. Download data on
  demand, and keep `data/` out of git.
- `kagglehub` caches downloads, so only the first run pays the download cost.

## Load the images into tensors

A naive loader looks like this:

```python
X_train = np.loadtxt(...)                         # (2000, 30000)
X_train = X_train.reshape(len(X_train), 100, 100, 3)
X_train = X_train / 255.
```

We make three practical improvements:

1. **`pandas.read_csv`** parses the wide CSV faster than `np.loadtxt`.
2. The reshaped `uint8` arrays are cached as `.npz` (~48 MB instead of 1.4 GB).
3. **Labels are not scaled.** They stay `0` / `1`.

The CSV path stores images as NHWC `(N, 100, 100, 3)`; `permute(0, 3, 1, 2)`
converts them to PyTorch's NCHW layout and divides by 255. The image-folder path
gets the same result from `transforms.ToTensor()`.

In [ ]:
# Build tensors and DataLoaders: NHWC uint8 -> NCHW float in [0, 1].
def load_csv_split(images_path, labels_path):
    """Read one flattened CSV split and cache a compact uint8 array."""
    cache_path = images_path.with_name(f"{images_path.stem}_{IMG_SIZE}px.npz")

    if cache_path.exists():
        cached = np.load(cache_path)
        return (
            np.array(cached["images"], copy=True),
            np.array(cached["labels"], copy=True),
        )

    print(f"Reading {images_path.name} ({images_path.stat().st_size / 1e9:.2f} GB) ...")
    images = pd.read_csv(images_path, header=None, dtype=np.float32).to_numpy()
    images = images.reshape(-1, IMG_SIZE, IMG_SIZE, 3).astype(np.uint8)

    labels = pd.read_csv(labels_path, header=None, dtype=np.int64).to_numpy().ravel()

    np.savez_compressed(cache_path, images=images, labels=labels)
    print(f"Cached -> {cache_path.name}")
    return images, labels


def make_tensor_dataset(images, labels):
    """uint8 NHWC -> float32 NCHW in [0, 1], with torch.long labels."""
    x = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2).div_(255)
    y = torch.tensor(labels, dtype=torch.long)
    return TensorDataset(x, y)


def dog_first(label):
    """ImageFolder sorts cats before dogs; flip so 0 = dog, 1 = cat."""
    return 1 - label


if CSV_ROOT is not None:
    train_images_path = find_csv(CSV_ROOT, "input.csv")
    train_labels_path = find_csv(CSV_ROOT, "labels.csv")
    test_images_path = find_csv(CSV_ROOT, "input_test.csv")
    test_labels_path = find_csv(CSV_ROOT, "labels_test.csv")

    X_train, y_train = load_csv_split(train_images_path, train_labels_path)
    X_test, y_test = load_csv_split(test_images_path, test_labels_path)

    print(f"\nX_train: {X_train.shape} | y_train: {y_train.shape}")
    print(f"X_test : {X_test.shape} | y_test : {y_test.shape}")
    print(f"Pixel range: {X_train.min()} to {X_train.max()}")

    train_dataset = make_tensor_dataset(X_train, y_train)
    test_dataset = make_tensor_dataset(X_test, y_test)

else:
    cats_dir = download_and_extract(PUBLIC_ZIP_URL, "cats_and_dogs_filtered")

    eval_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ])

    train_dataset = datasets.ImageFolder(
        cats_dir / "train", transform=eval_transform, target_transform=dog_first
    )
    test_dataset = datasets.ImageFolder(
        cats_dir / "validation", transform=eval_transform, target_transform=dog_first
    )

# Dedicated generator: reproducible shuffling even if other cells use the RNG.
train_generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"\nTrain images: {len(train_dataset):,} | Test images: {len(test_dataset):,}")
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")
print("Class -> label:", {name: i for i, name in enumerate(CLASS_NAMES)})

In [ ]:
# Sanity-check one batch and look at a few images before training.
images, labels = next(iter(train_loader))

print(f"One batch: images {tuple(images.shape)} | labels {tuple(labels.shape)}")
print(f"Image dtype: {images.dtype} | pixel range: {images.min():.2f} to {images.max():.2f}")
print(f"Label dtype: {labels.dtype}")
print("First 8 labels:", labels[:8].tolist())
print("First 8 names :", [CLASS_NAMES[i] for i in labels[:8]])


def show_samples(dataset, class_names, n=8, seed=42):
    random.seed(seed)
    indices = random.sample(range(len(dataset)), n)

    fig, axes = plt.subplots(2, n // 2, figsize=(14, 7))
    for ax, idx in zip(axes.ravel(), indices):
        image, label = dataset[idx]
        ax.imshow(image.permute(1, 2, 0).numpy())   # CHW -> HWC for plotting
        ax.set_title(f"{class_names[int(label)]} (label={int(label)})")
        ax.axis(False)

    plt.tight_layout()
    plt.show()


show_samples(train_dataset, CLASS_NAMES)

**Takeaways**

- The `(2000, 30000)` matrix becomes `(2000, 3, 100, 100)`.
- `ToTensor()` (or the explicit `/255`) scales pixels to `[0, 1]`; labels stay
  integer class indices.
- The first 1,000 CSV rows are dogs, then 1,000 cats. **Shuffle the training
  loader**, or each batch contains only one class.
- The public `ImageFolder` fallback is alphabetically ordered, so we remap its
  labels to preserve `0 = dog, 1 = cat`.
- A batch is `(32, 3, 100, 100)` with `(32,)` integer labels.

## Build the CNN

The architecture:

```
100x100x3
  Conv2d(3  -> 32, 3x3)  ->  98x98x32
  MaxPool2d(2)           ->  49x49x32
  Conv2d(32 -> 32, 3x3)  ->  47x47x32
  MaxPool2d(2)           ->  23x23x32
  Flatten                ->  16,928 values
  Linear(16,928 -> 64)   ->  64
  Linear(64 -> 1)        ->  1 raw logit
```

`nn.Linear` does not infer the flatten size, so `32 * 23 * 23` is written out.
A wrong number here is the most common CNN bug in PyTorch.

Most parameters live in the dense layer, not the convolutions. That is exactly
why convolutions are efficient: the same 3x3 filters are reused across the image.

In [ ]:
# Define the CNN; check parameter count and output shape before training.
class CatDogCNN(nn.Module):
    """A small CNN for 100x100 RGB images."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 23 * 23, 64),
            nn.ReLU(),
            nn.Linear(64, 1),              # one raw logit: no sigmoid here
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


torch.manual_seed(42)
model = CatDogCNN().to(device)

model_stats = summary(
    model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), device=device, verbose=0
)
print(model_stats)
print(f"\nTrainable parameters: {count_parameters(model):,}")

# Zeros confirm the forward shape without consuming the RNG.
model.eval()
with torch.inference_mode():
    dummy = model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device))
print("Output shape:", tuple(dummy.shape))

**Takeaways**

- The model outputs **one logit**, not a probability. `BCEWithLogitsLoss` applies
  the sigmoid internally.
- ~1.1 M parameters, almost all of them in the `Linear(16928, 64)` layer.
- `summary(...)` prints layer shapes and parameter counts; `print(model)` is the
  no-dependency fallback.
- A dummy forward pass catches shape mistakes before training starts.

## Loss, optimizer, and the training loop

Training is five explicit steps per batch:

```python
optimizer.zero_grad()      # clear old gradients
logits = model(images)     # forward pass
loss = loss_fn(logits, targets)
loss.backward()            # backpropagation
optimizer.step()           # update the weights
```

For binary classification with one output:

- **Loss:** `nn.BCEWithLogitsLoss()` — expects raw logits and float targets
  shaped `(N, 1)`.
- **Optimizer:** `optim.Adam(model.parameters(), lr=0.001)` — a common default.
- **Accuracy:** `sigmoid(logit) > 0.5`, compared with the integer label.

In [ ]:
# Helpers: one epoch of training, evaluation, and the loss/accuracy curves.
def binary_accuracy(logits, targets):
    return ((torch.sigmoid(logits) > 0.5) == targets.bool()).sum().item()


def evaluate_model(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            targets = labels.float().unsqueeze(1).to(device)
            logits = model(images)

            total_loss += loss_fn(logits, targets).item() * images.size(0)
            correct += binary_accuracy(logits, targets)
            total += images.size(0)

    return total_loss / total, correct / total


def train_model(model, loader, loss_fn, optimizer, epochs, test_loader):
    history = {
        "loss": [],
        "accuracy": [],
        "test_loss": [],
        "test_accuracy": [],
    }

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in loader:
            images = images.to(device)
            targets = labels.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += binary_accuracy(logits, targets)
            total += images.size(0)

        test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)

        history["loss"].append(running_loss / total)
        history["accuracy"].append(correct / total)
        history["test_loss"].append(test_loss)
        history["test_accuracy"].append(test_acc)

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"loss {history['loss'][-1]:.4f} | acc {history['accuracy'][-1] * 100:5.1f}% | "
            f"test loss {test_loss:.4f} | test acc {test_acc * 100:5.1f}%"
        )

    return history


def plot_history(history):
    epochs = range(1, len(history["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["loss"], label="train")
    axes[0].plot(epochs, history["test_loss"], label="test")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("BCE loss")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(epochs, history["accuracy"], label="train")
    axes[1].plot(epochs, history["test_accuracy"], label="test")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# Fit the model for the configured number of epochs.
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

history = train_model(
    model,
    train_loader,
    loss_fn,
    optimizer,
    epochs=EPOCHS,
    test_loader=test_loader,
)

In [ ]:
# Plot train vs test loss and accuracy.
plot_history(history)

**Takeaways**

On the offline CSVs this run reaches (the public fallback has 1,000 test images,
so its numbers will be close but not identical):

- **Train:** 89.1% accuracy, loss 0.265 at epoch 10.
- **Test:** 71.0% accuracy, loss 0.705 at epoch 10.
- Test loss bottoms out at **0.578 in epoch 8**, then rises to 0.705 while train
  loss keeps falling.

That widening gap is **overfitting**: the model is memorizing the training set.
Accuracy alone would not show it; the two loss curves do.

- 71% is well above the 50% random baseline, so the CNN learned real features.
- It is not a finished model. The next section shows where the errors are, and
  the recap lists what to try next.
- The run takes about 8 seconds per epoch on CPU (10 epochs ≈ 80 seconds).

## Evaluate the model

`model.evaluate()` does not exist in PyTorch, so `evaluate_model` plays that
role. One accuracy number is not enough: a model can be good at one class and
bad at the other.

The next cells compute:

- **Precision** — when the model says "dog", how often is it right?
- **Recall** — of all real dogs, how many did it find?
- **F1** — a balance between the two.
- A **confusion matrix** — exactly which classes get mixed up.

In [ ]:
# Evaluate on the test set and turn logits into class predictions.
test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc * 100:.1f}%")
print(f"Trainable parameters: {count_parameters(model):,}")


def predict_logits(model, loader):
    model.eval()
    all_logits = []
    with torch.inference_mode():
        for images, _ in loader:
            all_logits.append(model(images.to(device)).cpu())
    return torch.cat(all_logits)


logits = predict_logits(model, test_loader)
probabilities = torch.sigmoid(logits).squeeze(1)
y_pred = (probabilities > 0.5).long().numpy()

# shuffle=False keeps the loader order aligned with the predictions.
y_true = torch.cat([labels for _, labels in test_loader]).numpy()

print("Logits shape:", tuple(logits.shape))
print("First 5 probabilities:", [round(p, 3) for p in probabilities[:5].tolist()])

In [ ]:
# Per-class metrics and a confusion matrix reveal class bias.
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

plt.figure(figsize=(6, 5))
sns.heatmap(
    confusion_matrix(y_true, y_pred),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix — test set")
plt.show()

**Takeaways**

The numbers below are from the offline CSV run (400 test images). The public
fallback uses 1,000 test images, so its report will differ slightly.

- Test accuracy is **71.0%**, but the classes are not treated equally:
  - dog recall **0.545** (109 / 200 dogs found)
  - cat recall **0.875** (175 / 200 cats found)
  - dog precision **0.813**, cat precision **0.658**
- The confusion matrix is `[[109, 91], [25, 175]]`: 91 dogs are called cats,
  but only 25 cats are called dogs. The model is biased toward "cat".
- Accuracy can hide that bias. Precision, recall and the confusion matrix expose it.
- A 50/50 dataset also means a lazy "always dog" model would score 50%; 71% is
  better than that, but not by enough to ignore the errors.

## Predict on individual images

To predict one image, the preprocessing must match training exactly:

1. Load it as RGB.
2. Resize to `100x100`.
3. Scale to `[0, 1]` and switch to `(C, H, W)`.
4. Add a batch dimension: `(3, 100, 100) -> (1, 3, 100, 100)`.

The model returns one logit. `sigmoid(logit)` gives the probability of class 1
(cat); `> 0.5` maps it to a class name. The threshold is adjustable — 0.5 is a
default, not a law.

In [ ]:
# Predict individual images; sigmoid > 0.5 means cat.
def predict_one(model, image_tensor):
    """Return (predicted index, probability of class 1) for one CHW image."""
    model.eval()
    with torch.inference_mode():
        logit = model(image_tensor.unsqueeze(0).to(device))

    probability = torch.sigmoid(logit).item()
    return int(probability > 0.5), probability


random.seed(7)
indices = random.sample(range(len(test_dataset)), 8)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.ravel(), indices):
    image, label = test_dataset[idx]
    predicted, probability = predict_one(model, image)

    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(
        f"true: {CLASS_NAMES[int(label)]}\n"
        f"pred: {CLASS_NAMES[predicted]} ({probability:.2f})"
    )
    ax.axis(False)

plt.tight_layout()
plt.show()

**Takeaways**

- The model always expects a batch: `unsqueeze(0)` adds the missing `N`.
- One logit → `sigmoid` → probability → threshold at `0.5`.
- A confidence score is not a guarantee. Several of these predictions are wrong.
- Always preprocess inference images exactly like the training images. A mismatch
  (wrong size, wrong channel order, no scaling) makes predictions meaningless.

## Save and reload the model

Save the **`state_dict`** (the weights), not the whole model object:

1. `torch.save(model.state_dict(), path)`
2. Rebuild the same architecture.
3. `model.load_state_dict(torch.load(path))`
4. Call `model.eval()` before inference.

The architecture must match exactly, and `map_location=device` lets a GPU-trained
file load on a CPU machine.

In [ ]:
# Save weights, rebuild the architecture, and check the reload.
SAVE_DIR = repo_root() / "cat-vs-dog" / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = SAVE_DIR / "cat_dog_cnn.pth"

torch.save(model.state_dict(), MODEL_PATH)
print("Saved weights:", MODEL_PATH)

loaded_model = CatDogCNN().to(device)
loaded_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device, weights_only=True)
)
loaded_model.eval()

before_loss, before_acc = evaluate_model(model, test_loader, loss_fn)
loaded_loss, loaded_acc = evaluate_model(loaded_model, test_loader, loss_fn)

print(f"Before save: loss {before_loss:.4f} | accuracy {before_acc * 100:.1f}%")
print(f"Reloaded   : loss {loaded_loss:.4f} | accuracy {loaded_acc * 100:.1f}%")

**Takeaways**

- Save weights only: smaller, portable, and explicit to reload.
- Rebuild the architecture before `load_state_dict`.
- `model.eval()` matters for dropout and batch-norm; it is a no-op here but is a
  habit worth keeping.
- The reloaded model should produce exactly the same metrics.

## High-level learnings

The exact figures below are from the local CSV run; the public fallback has the
same shape of curve with slightly different values.

| Learning | What to remember |
|---|---|
| Data pipeline first | Loading, reshaping, scaling and batching take more code than the model. Get this right before tuning. |
| Labels are not pixels | Scale images to `[0,1]`, never the labels. Scaling labels destroys the targets and silently breaks training. |
| NCHW vs NHWC | PyTorch wants `(N, C, H, W)`. `ToTensor()` and `permute` do the conversion. |
| Dataset → DataLoader | `TensorDataset` / `ImageFolder` hold the data; `DataLoader` batches and shuffles it. Always shuffle training. |
| Convolutions are efficient | The same small filters slide across the image. The dense layer, not the convolutions, holds most parameters. |
| Raw logits + loss | One output logit + `BCEWithLogitsLoss`; apply `sigmoid` only at inference. |
| Overfitting is the default | Train loss fell to 0.265 while test loss turned up at 0.578 → 0.705. Watch the gap, not just accuracy. |
| Accuracy hides class bias | The confusion matrix showed 91 dogs predicted as cats vs 25 cats predicted as dogs. |
| Reproducibility | Seed, `model.train()` / `model.eval()`, `inference_mode`, and `state_dict` are part of the workflow. |
| Data does not belong in git | 1.4 GB exceeds GitHub's 100 MB per-file limit. Download on demand and ignore `data/`. |

### If you only remember three things

1. **Data:** `(N, 100, 100, 3)` → `(N, 3, 100, 100)` floats in `[0,1]`, labels
   `0/1` as `torch.long`.
2. **Model:** convolution → ReLU → pool → flatten → dense → **one logit**;
   train with `BCEWithLogitsLoss`.
3. **Evaluation:** compare train vs test curves, then use a classification report
   and confusion matrix. One accuracy number is not enough.

## Next experiments

The baseline is deliberately simple. These are the usual next moves, roughly in
order of payoff:

1. **Data augmentation** — `RandomHorizontalFlip`, `RandomResizedCrop`,
   `RandomRotation` on the **training** set only. This is the cheapest way to
   fight the overfitting seen above.
2. **Dropout** — add `nn.Dropout(0.3)` before the final linear layer.
3. **A real validation split** — hold out part of training data, use it for early
   stopping, and keep the test set untouched.
4. **Early stopping** — stop when validation loss stops improving (test loss
   already turned up at epoch 9).
5. **More data** — the full Kaggle dataset has ~25,000 images; 2,000 is small.
6. **Transfer learning** — start from a pretrained ResNet/EfficientNet and
   fine-tune the last layers. This usually beats a small CNN trained from scratch.
7. **Threshold tuning** — 0.5 is not sacred. If false cats are expensive, move it.
8. **Class weights** — only needed if the classes are imbalanced; this dataset is
   balanced, so fix the model before adding weights.

The next notebook in this repository, `04_convolutional_neural_networks.ipynb`,
walks through augmentation, pooling and transfer-learning-style experiments in
more detail.

## Exercises

1. Set `EPOCHS = 1` and run the notebook end-to-end. Which cells are slowest?
2. Add `nn.Dropout(0.3)` before the final `Linear`. Does test accuracy improve?
3. Add `transforms.RandomHorizontalFlip()` to the training data only. Retrain.
   Compare the train/test gap with the baseline.
4. Split the training set into train/validation and implement early stopping with
   `patience=3`.
5. Change the decision threshold from `0.5` to `0.3` and `0.7`. How do precision
   and recall for "cat" change?
6. Use `kagglehub` to download the exact Kaggle dataset and compare its test score
   with the public fallback. Why are the numbers not identical?

## Extra resources and data credits

- [PyTorch CNN tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)
- [CNN Explainer](https://poloclub.github.io/cnn-explainer/) — interactive TinyVGG
- [A guide to convolution arithmetic](https://arxiv.org/pdf/1603.07285.pdf) — padding, stride, output sizes
- [PyTorch optimization loop](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

**Dataset**

- Exact CSVs: [Kaggle — Cats & Dogs images](https://www.kaggle.com/datasets/tarekmohamedadel/cats-and-dogs-images)
  (Apache 2.0), downloaded with `kagglehub`.
- Public fallback: [cats_and_dogs_filtered mirror](https://github.com/MLBasic/cats_and_dogs_filtered)
  (2,000 train / 1,000 validation images).
- GitHub large-file limits: [About large files on GitHub](https://docs.github.com/en/repositories/working-with-files/managing-large-files/about-large-files-on-github).